# Notebook 04 — Decision Engine and Validation for Predicted Late-Stage Clone Selection

## Goal

This notebook converts **predicted late-stage outcomes** from Notebook 03b into an actual clone selection decision.

We assume:

- at selection time, **true late outcomes are unknown**
- only **early-stage measurements** are available in real deployment
- Notebook 03b provides predicted late-stage qP, qP drop, aggregation, and stability probability
- Notebook 04b applies selection rules to recommend clones

---

## Why this notebook exists

Notebook 03b is the **prediction engine**:
- it predicts future clone performance

Notebook 04b is the **decision engine**:
- it applies operational rules
- filters risky clones
- ranks the remaining candidates
- simulates how clone selection would work in practice

This separation is useful because:
- prediction and decision are different tasks
- thresholds can change without retraining models
- future SDL systems may add rescue / process-control logic here

---

## Current decision logic

We currently use this deployment-style selection framework:

1. **Quality filter**
   - remove clones with predicted late aggregation above threshold

2. **Stability filter**
   - remove clones with low predicted stability probability

3. **Productivity ranking**
   - among remaining clones, rank by predicted late qP

---

## Decision modes

We compare two modes:

### A. Biosimilar mode
- productivity is emphasized
- stability is important
- aggregation penalty is lighter

### B. Novel / ADC mode
- quality is emphasized more strongly
- stability remains important
- productivity still matters, but quality threshold is stricter

---

## Important note

All `true_*` columns are used **only for offline evaluation in synthetic data**.

In real deployment:
- only `pred_*` columns would be available

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

scenario = "legacy"   # or "optimized"
n_clones = 5000

TOP_N = 10
TOP_STAGE2 = 3

print("Scenario:", scenario)
print("n_clones:", n_clones)
print("Top-N final:", TOP_N)
print("Top-N stage2:", TOP_STAGE2)

Scenario: legacy
n_clones: 5000
Top-N final: 10
Top-N stage2: 3


## Step 1 — Load prediction table from Notebook 03b

Here we load the prediction table produced in Notebook 03b.

This table should already contain:

- predicted late qP
- predicted qP drop
- predicted late aggregation
- predicted stable probability
- true late outcomes for offline simulation only

In [2]:
PRED_PATH = Path("../data/synthetic/processed") / f"predictions_03b_qp_{n_clones}_{scenario}.csv"
print("Loading predictions:", PRED_PATH)

df = pd.read_csv(PRED_PATH)
print("Shape:", df.shape)
display(df.head())

Loading predictions: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 12)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_1502,0.423727,2.782248e-08,9.245089,0.437072,0,0.520022,2.236766e-08,6.841477,0,0.001451,0.000000
1,CLONE_2587,0.452226,7.390266e-08,3.054489,0.180526,0,0.647150,4.849476e-08,4.346930,0,0.000000,0.006281
2,CLONE_2654,0.455506,2.763576e-08,8.264028,0.251498,0,0.460908,2.481724e-08,9.193986,0,0.001538,0.000000
3,CLONE_1056,0.431212,6.228160e-08,7.100926,0.227443,0,0.390078,6.751316e-08,3.772907,0,0.003182,0.000000
4,CLONE_0706,0.623201,1.823740e-07,8.968594,0.005890,0,0.680538,1.487222e-07,7.045437,0,0.000000,0.006086


## Step 2 — Check schema compatibility

Before applying selection rules, we verify that the prediction table matches the expected 03b output schema.

This prevents silent notebook mismatch caused by:
- outdated file names
- old column naming
- partial notebook updates

In [3]:
required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_stable_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_stable_label",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)
assert len(missing) == 0, f"04b is missing required columns: {missing}"

optional_cols = ["pred_super_prob", "pred_aggr_prob"]
present_optional = [c for c in optional_cols if c in df.columns]
print("Optional columns found:", present_optional)

Missing required columns: []
Optional columns found: ['pred_super_prob', 'pred_aggr_prob']


## Step 3 — Define helper functions

We use helper functions for:

- z-score normalization
- ranking utility calculation
- top-k overlap
- elite capture metrics

These help compare decision quality across selection modes.

In [4]:
def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def make_pred_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["pred_late_qp"])
        - w_drop * z(df["pred_qp_drop"])
        - w_agg * z(df["pred_late_agg"])
    )

def make_true_utility(df, w_qp, w_drop, w_agg):
    return (
        w_qp * z(df["true_late_qp"])
        - w_drop * z(df["true_qp_drop"])
        - w_agg * z(df["true_late_agg"])
    )

def topk_overlap(true_scores, pred_scores, k):
    true_top = set(pd.Series(true_scores).nlargest(k).index)
    pred_top = set(pd.Series(pred_scores).nlargest(k).index)
    return len(true_top & pred_top) / k

def precision_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    return float(top[elite_col].mean())

def ndcg_at_k_df(df, score_col, elite_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    rel = top[elite_col].values.astype(float)
    discounts = 1.0 / np.log2(np.arange(2, len(rel) + 2))
    dcg = float(np.sum(rel * discounts))
    ideal = np.sort(rel)[::-1]
    idcg = float(np.sum(ideal * discounts)) + 1e-12
    return dcg / idcg

## Step 4 — Define offline evaluation utilities

Because this is a synthetic dataset, we can define an offline “true utility” score
to evaluate how good the selected clones really are.

This is **evaluation only** and would not be available in real deployment.

In [5]:
# biosimilar-style evaluation utility
W_BIO = dict(w_qp=1.0, w_drop=1.0, w_agg=0.2)

# novel / ADC-style evaluation utility
W_NOVEL = dict(w_qp=1.0, w_drop=1.0, w_agg=1.0)

df["true_util_bio"] = make_true_utility(df, **W_BIO)
df["true_util_novel"] = make_true_utility(df, **W_NOVEL)

df["pred_util_bio"] = make_pred_utility(df, **W_BIO)
df["pred_util_novel"] = make_pred_utility(df, **W_NOVEL)

display(df[[
    "clone_id",
    "true_util_bio", "pred_util_bio",
    "true_util_novel", "pred_util_novel"
]].head())

,clone_id,true_util_bio,pred_util_bio,true_util_novel,pred_util_novel
0,CLONE_1502,-0.535924,-0.313414,-0.631460,-1.176414
1,CLONE_2587,-1.153387,0.021145,-0.609472,1.021513
2,CLONE_2654,-0.322472,-0.523026,-1.021048,-1.090728
3,CLONE_1056,0.468836,-0.127133,1.159895,-0.344741
4,CLONE_0706,-1.512834,-1.657913,-1.660653,-2.437689


## Step 5A — Decision mode: Biosimilar

In biosimilar mode, the selection logic is:

1. remove clones with poor predicted quality
2. remove clones with low predicted stability probability
3. rank the remaining clones by predicted late qP

This reflects a setting where:
- productivity matters strongly
- stability matters strongly
- aggregation still matters, but slightly less than in novel mode

In [6]:
# --------------------------------------------------
# Step 5A — Biosimilar 3-bucket decision engine
# --------------------------------------------------

BIO_AGG_MAX = 15.0
BIO_STABLE_PROB_MIN = 0.50

# rescue margin (핵심 파라미터)
MARGIN_AGG = 3.0
MARGIN_STABLE = 0.10

bio = df.copy()

# ---------------------------
# Bucket definitions
# ---------------------------

# PASS (strict)
bio["pass_quality"] = bio["pred_late_agg"] <= BIO_AGG_MAX
bio["pass_stability"] = bio["pred_stable_prob"] >= BIO_STABLE_PROB_MIN
bio["bucket_pass"] = bio["pass_quality"] & bio["pass_stability"]

# RESCUE (borderline)
bio["near_quality"] = bio["pred_late_agg"].between(BIO_AGG_MAX, BIO_AGG_MAX + MARGIN_AGG)
bio["near_stability"] = bio["pred_stable_prob"].between(
    BIO_STABLE_PROB_MIN - MARGIN_STABLE,
    BIO_STABLE_PROB_MIN
)

bio["high_prod"] = bio["pred_late_qp"] >= bio["pred_late_qp"].quantile(0.80)

bio["bucket_rescue"] = (
    (~bio["bucket_pass"]) &
    bio["high_prod"] &
    (bio["near_quality"] | bio["near_stability"])
)

# FAIL
bio["bucket_fail"] = ~(bio["bucket_pass"] | bio["bucket_rescue"])

# ---------------------------
# Ranking
# ---------------------------
bio_pass = bio[bio["bucket_pass"]].copy()
bio_rescue = bio[bio["bucket_rescue"]].copy()

bio_pass["rank_metric"] = bio_pass["pred_late_qp"]
bio_rescue["rank_metric"] = bio_rescue["pred_late_qp"]

bio_pass = bio_pass.sort_values("rank_metric", ascending=False)
bio_rescue = bio_rescue.sort_values("rank_metric", ascending=False)

# ---------------------------
# Stage selection
# ---------------------------
P10_PASS_N = 120
P10_RESCUE_N = 30

bio_stage1_pass = bio_pass.head(P10_PASS_N)
bio_stage1_rescue = bio_rescue.head(P10_RESCUE_N)

print("=== Biosimilar Stage 1 (P10) ===")
print("Pass:", len(bio_stage1_pass))
print("Rescue:", len(bio_stage1_rescue))
print("Fail:", int(bio["bucket_fail"].sum()))

display(bio_stage1_pass.head())
display(bio_stage1_rescue.head())

=== Biosimilar Stage 1 (P10) ===
Pass: 92
Rescue: 9
Fail: 899


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,...,pred_util_novel,pass_quality,pass_stability,bucket_pass,near_quality,near_stability,high_prod,bucket_rescue,bucket_fail,rank_metric
314,CLONE_1619,0.243030,2.196463e-06,3.403038,0.740186,1,0.153735,2.452274e-06,4.494834,1,...,8.393468,True,True,True,False,False,True,False,False,2.196463e-06
476,CLONE_2759,0.280103,9.707880e-07,12.075904,0.627099,1,0.064377,1.205105e-06,14.293786,1,...,1.543105,True,True,True,False,False,True,False,False,9.707880e-07
898,CLONE_3393,0.308014,7.794316e-07,8.439185,0.558652,1,0.365885,7.938732e-07,5.284487,0,...,2.154401,True,True,True,False,False,True,False,False,7.794316e-07
490,CLONE_0355,0.310332,5.067278e-07,7.402139,0.620693,1,0.421933,4.564358e-07,7.810760,0,...,1.799261,True,True,True,False,False,True,False,False,5.067278e-07
961,CLONE_1035,0.335483,4.939141e-07,8.600502,0.540727,1,0.434488,4.534849e-07,8.809331,0,...,1.090389,True,True,True,False,False,True,False,False,4.939141e-07


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,...,pred_util_novel,pass_quality,pass_stability,bucket_pass,near_quality,near_stability,high_prod,bucket_rescue,bucket_fail,rank_metric
115,CLONE_3602,0.349981,4.613437e-07,11.027346,0.422086,0,0.370824,4.773866e-07,12.370305,0,...,-0.038340,True,False,False,False,True,True,True,False,4.613437e-07
966,CLONE_0151,0.344520,3.820308e-07,5.656948,0.472694,0,0.450622,3.087592e-07,6.537014,0,...,1.820119,True,False,False,False,True,True,True,False,3.820308e-07
537,CLONE_3608,0.368305,2.869174e-07,2.250324,0.443046,0,0.400536,2.740198e-07,2.406392,0,...,2.637339,True,False,False,False,True,True,True,False,2.869174e-07
534,CLONE_3750,0.340166,2.611024e-07,3.087452,0.423077,0,0.441202,2.247222e-07,4.907386,0,...,2.504301,True,False,False,False,True,True,True,False,2.611024e-07
719,CLONE_3186,0.339105,2.240807e-07,5.541740,0.487078,0,0.431345,1.980545e-07,3.021272,0,...,1.491938,True,False,False,False,True,True,True,False,2.240807e-07


In [7]:
# --------------------------------------------------
# Stage 2 (P15) refinement
# --------------------------------------------------

# Combine pass + rescue from Stage 1
stage2_pool = pd.concat([bio_stage1_pass, bio_stage1_rescue], ignore_index=True)

# re-rank (같은 metric 유지 or utility 써도 OK)
stage2_pool["rank_metric"] = stage2_pool["pred_late_qp"]

stage2_pool = stage2_pool.sort_values("rank_metric", ascending=False)

P15_PASS_N = 25
P15_RESCUE_N = 8

stage2_pass = stage2_pool.head(P15_PASS_N)
stage2_rescue = stage2_pool.iloc[P15_PASS_N:P15_PASS_N + P15_RESCUE_N]

print("=== Stage 2 (P15) ===")
print("Pass:", len(stage2_pass))
print("Rescue:", len(stage2_rescue))

display(stage2_pass.head())
display(stage2_rescue.head())

=== Stage 2 (P15) ===
Pass: 25
Rescue: 8


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,...,pred_util_novel,pass_quality,pass_stability,bucket_pass,near_quality,near_stability,high_prod,bucket_rescue,bucket_fail,rank_metric
0,CLONE_1619,0.243030,2.196463e-06,3.403038,0.740186,1,0.153735,2.452274e-06,4.494834,1,...,8.393468,True,True,True,False,False,True,False,False,2.196463e-06
1,CLONE_2759,0.280103,9.707880e-07,12.075904,0.627099,1,0.064377,1.205105e-06,14.293786,1,...,1.543105,True,True,True,False,False,True,False,False,9.707880e-07
2,CLONE_3393,0.308014,7.794316e-07,8.439185,0.558652,1,0.365885,7.938732e-07,5.284487,0,...,2.154401,True,True,True,False,False,True,False,False,7.794316e-07
3,CLONE_0355,0.310332,5.067278e-07,7.402139,0.620693,1,0.421933,4.564358e-07,7.810760,0,...,1.799261,True,True,True,False,False,True,False,False,5.067278e-07
4,CLONE_1035,0.335483,4.939141e-07,8.600502,0.540727,1,0.434488,4.534849e-07,8.809331,0,...,1.090389,True,True,True,False,False,True,False,False,4.939141e-07


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,...,pred_util_novel,pass_quality,pass_stability,bucket_pass,near_quality,near_stability,high_prod,bucket_rescue,bucket_fail,rank_metric
94,CLONE_3608,0.368305,2.869174e-07,2.250324,0.443046,0,0.400536,2.740198e-07,2.406392,0,...,2.637339,True,False,False,False,True,True,True,False,2.869174e-07
23,CLONE_4507,0.326598,2.721100e-07,7.816647,0.506496,1,0.288739,2.956017e-07,5.788807,1,...,0.875003,True,True,True,False,False,True,False,False,2.721100e-07
95,CLONE_3750,0.340166,2.611024e-07,3.087452,0.423077,0,0.441202,2.247222e-07,4.907386,0,...,2.504301,True,False,False,False,True,True,True,False,2.611024e-07
24,CLONE_0645,0.293886,2.569627e-07,5.571826,0.635621,1,0.486411,1.991099e-07,4.477654,0,...,1.970608,True,True,True,False,False,True,False,False,2.569627e-07
25,CLONE_1783,0.278180,2.563572e-07,7.575139,0.734982,1,0.313885,2.545850e-07,7.023149,0,...,1.355098,True,True,True,False,False,True,False,False,2.563572e-07


In [8]:
# --------------------------------------------------
# Final selection from Stage 2 pool
# --------------------------------------------------

FINAL_TOP_N = TOP_N      # usually 10
FINAL_TOP3 = TOP_STAGE2  # usually 3

final_pool = pd.concat([stage2_pass, stage2_rescue], ignore_index=True)
final_pool = final_pool.sort_values(
    ["pred_late_qp", "pred_stable_prob"],
    ascending=[False, False]
).copy()

bio_top10 = final_pool.head(FINAL_TOP_N).copy()
bio_top3 = final_pool.head(FINAL_TOP3).copy()

print("=== Biosimilar Final Selection ===")
print("Final top10:", len(bio_top10))
print("Final top3:", len(bio_top3))

display(bio_top10[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in bio_top10.columns]])

display(bio_top3[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in bio_top3.columns]])

=== Biosimilar Final Selection ===
Final top10: 10
Final top3: 3


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_1619,2.196463e-06,0.243030,3.403038,0.740186,2.452274e-06,0.153735,4.494834,1,0.922108,0.002972
1,CLONE_2759,9.707880e-07,0.280103,12.075904,0.627099,1.205105e-06,0.064377,14.293786,1,0.668066,0.001667
2,CLONE_3393,7.794316e-07,0.308014,8.439185,0.558652,7.938732e-07,0.365885,5.284487,0,0.566006,0.000000
3,CLONE_0355,5.067278e-07,0.310332,7.402139,0.620693,4.564358e-07,0.421933,7.810760,0,0.215980,0.000000
4,CLONE_1035,4.939141e-07,0.335483,8.600502,0.540727,4.534849e-07,0.434488,8.809331,0,0.291253,0.000000
5,CLONE_4119,4.807651e-07,0.332187,3.761302,0.525444,4.881243e-07,0.299740,2.908559,1,0.240372,0.000000
6,CLONE_2358,4.670190e-07,0.294368,5.048017,0.666567,5.567181e-07,0.202139,4.869253,1,0.174645,0.000000
7,CLONE_3602,4.613437e-07,0.349981,11.027346,0.422086,4.773866e-07,0.370824,12.370305,0,0.239048,0.001512
8,CLONE_4484,4.013054e-07,0.336678,3.564282,0.567257,4.399181e-07,0.319189,4.874848,0,0.230634,0.000000
9,CLONE_0151,3.820308e-07,0.344520,5.656948,0.472694,3.087592e-07,0.450622,6.537014,0,0.055487,0.000000


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_1619,2.196463e-06,0.243030,3.403038,0.740186,2.452274e-06,0.153735,4.494834,1,0.922108,0.002972
1,CLONE_2759,9.707880e-07,0.280103,12.075904,0.627099,1.205105e-06,0.064377,14.293786,1,0.668066,0.001667
2,CLONE_3393,7.794316e-07,0.308014,8.439185,0.558652,7.938732e-07,0.365885,5.284487,0,0.566006,0.000000


In [9]:
def eval_selection(df_sel, label="selection"):
    return {
        "label": label,
        "n": len(df_sel),
        "mean_true_late_qp": float(df_sel["true_late_qp"].mean()),
        "mean_true_qp_drop": float(df_sel["true_qp_drop"].mean()),
        "mean_true_late_agg": float(df_sel["true_late_agg"].mean()),
        "stable_rate": float(df_sel["true_stable_label"].mean()),
    }

bio_eval_table = pd.DataFrame([
    eval_selection(bio_stage1_pass, "bio_P10_pass"),
    eval_selection(bio_stage1_rescue, "bio_P10_rescue"),
    eval_selection(stage2_pass, "bio_P15_pass"),
    eval_selection(stage2_rescue, "bio_P15_rescue"),
    eval_selection(bio_top10, "bio_final_top10"),
    eval_selection(bio_top3, "bio_final_top3"),
])

display(bio_eval_table)

bio_eval = {
    "selected_n": len(bio_top10),
    "mean_true_late_qp": float(bio_top10["true_late_qp"].mean()),
    "mean_true_qp_drop": float(bio_top10["true_qp_drop"].mean()),
    "mean_true_late_agg": float(bio_top10["true_late_agg"].mean()),
    "stable_rate": float(bio_top10["true_stable_label"].mean()),
    "top10_overlap_true_util_bio": topk_overlap(df["true_util_bio"], df["pred_util_bio"], TOP_N),
}
pd.DataFrame([bio_eval])

,label,n,mean_true_late_qp,mean_true_qp_drop,mean_true_late_agg,stable_rate
0,bio_P10_pass,92,2.289683e-07,0.312750,6.286374,0.478261
1,bio_P10_rescue,9,2.542160e-07,0.404499,6.667635,0.000000
2,bio_P15_pass,25,5.135341e-07,0.285054,6.637278,0.440000
3,bio_P15_rescue,8,2.562187e-07,0.336362,4.783443,0.375000
4,bio_final_top10,10,7.632078e-07,0.308293,7.225318,0.400000
5,bio_final_top3,3,1.483750e-06,0.194666,8.024369,0.666667


,selected_n,mean_true_late_qp,mean_true_qp_drop,mean_true_late_agg,stable_rate,top10_overlap_true_util_bio
0,10,7.632078e-07,0.308293,7.225318,0.4,0.4


## Step 6A — Novel / ADC 3-bucket decision engine

In novel / ADC mode, quality is emphasized more strongly than in biosimilar mode.

So compared with biosimilar mode:

- aggregation threshold is stricter
- stability threshold is slightly stricter
- rescue margin is narrower
- productivity still matters, but low-quality clones are filtered more aggressively

We use the same 3-bucket logic:

- **pass** = strong candidates that clearly meet the criteria
- **rescue** = borderline but potentially valuable clones
- **fail** = clones that should be dropped

In [10]:
# --------------------------------------------------
# Step 6A — Novel / ADC 3-bucket decision engine
# --------------------------------------------------

NOVEL_AGG_MAX = 10.0
NOVEL_STABLE_PROB_MIN = 0.55

# rescue margin (stricter than biosimilar)
NOVEL_MARGIN_AGG = 2.0
NOVEL_MARGIN_STABLE = 0.08

novel = df.copy()

# ---------------------------
# Bucket definitions
# ---------------------------

# PASS (strict)
novel["pass_quality"] = novel["pred_late_agg"] <= NOVEL_AGG_MAX
novel["pass_stability"] = novel["pred_stable_prob"] >= NOVEL_STABLE_PROB_MIN
novel["bucket_pass"] = novel["pass_quality"] & novel["pass_stability"]

# RESCUE (borderline)
novel["near_quality"] = novel["pred_late_agg"].between(NOVEL_AGG_MAX, NOVEL_AGG_MAX + NOVEL_MARGIN_AGG)
novel["near_stability"] = novel["pred_stable_prob"].between(
    NOVEL_STABLE_PROB_MIN - NOVEL_MARGIN_STABLE,
    NOVEL_STABLE_PROB_MIN
)

novel["high_prod"] = novel["pred_late_qp"] >= novel["pred_late_qp"].quantile(0.85)

novel["bucket_rescue"] = (
    (~novel["bucket_pass"]) &
    novel["high_prod"] &
    (novel["near_quality"] | novel["near_stability"])
)

# FAIL
novel["bucket_fail"] = ~(novel["bucket_pass"] | novel["bucket_rescue"])

# ---------------------------
# Ranking
# ---------------------------
novel_pass = novel[novel["bucket_pass"]].copy()
novel_rescue = novel[novel["bucket_rescue"]].copy()

novel_pass["rank_metric"] = novel_pass["pred_late_qp"]
novel_rescue["rank_metric"] = novel_rescue["pred_late_qp"]

novel_pass = novel_pass.sort_values("rank_metric", ascending=False)
novel_rescue = novel_rescue.sort_values("rank_metric", ascending=False)

# ---------------------------
# Stage 1 selection (P10)
# ---------------------------
NOVEL_P10_PASS_N = 120
NOVEL_P10_RESCUE_N = 20

novel_stage1_pass = novel_pass.head(NOVEL_P10_PASS_N)
novel_stage1_rescue = novel_rescue.head(NOVEL_P10_RESCUE_N)

print("=== Novel / ADC Stage 1 (P10) ===")
print("Pass:", len(novel_stage1_pass))
print("Rescue:", len(novel_stage1_rescue))
print("Fail:", int(novel["bucket_fail"].sum()))

display(novel_stage1_pass[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_stage1_pass.columns]])

display(novel_stage1_rescue[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_stage1_rescue.columns]])

=== Novel / ADC Stage 1 (P10) ===
Pass: 53
Rescue: 20
Fail: 918


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
314,CLONE_1619,2.196463e-06,0.243030,3.403038,0.740186,2.452274e-06,0.153735,4.494834,1,0.922108,0.002972
898,CLONE_3393,7.794316e-07,0.308014,8.439185,0.558652,7.938732e-07,0.365885,5.284487,0,0.566006,0.000000
490,CLONE_0355,5.067278e-07,0.310332,7.402139,0.620693,4.564358e-07,0.421933,7.810760,0,0.215980,0.000000
206,CLONE_2358,4.670190e-07,0.294368,5.048017,0.666567,5.567181e-07,0.202139,4.869253,1,0.174645,0.000000
263,CLONE_4484,4.013054e-07,0.336678,3.564282,0.567257,4.399181e-07,0.319189,4.874848,0,0.230634,0.000000
134,CLONE_3969,3.813413e-07,0.294097,5.587836,0.725320,4.550859e-07,0.189066,5.380682,1,0.339964,0.000000
843,CLONE_0834,3.388848e-07,0.327803,6.191187,0.552750,3.740243e-07,0.309650,8.462860,0,0.114875,0.000000
60,CLONE_2378,3.386529e-07,0.328407,3.721325,0.568093,3.347188e-07,0.307016,4.670409,0,0.272585,0.000000
690,CLONE_4335,3.366401e-07,0.284232,9.790634,0.750745,3.453973e-07,0.300386,10.147165,0,0.363452,0.000000
477,CLONE_1757,3.344282e-07,0.281950,2.622051,0.730765,4.355070e-07,0.088379,1.773135,1,0.468231,0.000000


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
505,CLONE_0048,4.788019e-06,0.453842,10.695098,0.220395,1.751650e-05,0.523130,11.123578,0,0.698210,0.019093
730,CLONE_4878,3.345378e-06,0.497117,10.187778,0.176392,4.368307e-06,0.581270,9.401341,0,0.615499,0.016247
965,CLONE_4665,1.320135e-06,0.704642,11.574957,0.061885,9.387567e-07,0.739219,9.048022,0,0.170636,0.097010
801,CLONE_1505,5.616570e-07,0.519735,10.217237,0.010778,6.091065e-07,0.449628,11.902859,0,0.029747,0.004322
961,CLONE_1035,4.939141e-07,0.335483,8.600502,0.540727,4.534849e-07,0.434488,8.809331,0,0.291253,0.000000
251,CLONE_4119,4.807651e-07,0.332187,3.761302,0.525444,4.881243e-07,0.299740,2.908559,1,0.240372,0.000000
653,CLONE_2532,4.613569e-07,0.424645,10.106089,0.142804,4.798185e-07,0.420507,8.787420,0,0.065856,0.003019
115,CLONE_3602,4.613437e-07,0.349981,11.027346,0.422086,4.773866e-07,0.370824,12.370305,0,0.239048,0.001512
62,CLONE_4258,4.187108e-07,0.660721,10.237109,0.017243,4.168977e-07,0.665151,10.876121,0,0.021973,0.066060
966,CLONE_0151,3.820308e-07,0.344520,5.656948,0.472694,3.087592e-07,0.450622,6.537014,0,0.055487,0.000000


In [11]:
# --------------------------------------------------
# Novel / ADC Stage 2 (P15) refinement
# --------------------------------------------------

novel_stage2_pool = pd.concat([novel_stage1_pass, novel_stage1_rescue], ignore_index=True)

novel_stage2_pool["rank_metric"] = novel_stage2_pool["pred_late_qp"]
novel_stage2_pool = novel_stage2_pool.sort_values(
    ["rank_metric", "pred_stable_prob"],
    ascending=[False, False]
).copy()

NOVEL_P15_PASS_N = 25
NOVEL_P15_RESCUE_N = 5

novel_stage2_pass = novel_stage2_pool.head(NOVEL_P15_PASS_N).copy()
novel_stage2_rescue = novel_stage2_pool.iloc[
    NOVEL_P15_PASS_N:NOVEL_P15_PASS_N + NOVEL_P15_RESCUE_N
].copy()

print("=== Novel / ADC Stage 2 (P15) ===")
print("Pass:", len(novel_stage2_pass))
print("Rescue:", len(novel_stage2_rescue))

display(novel_stage2_pass[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_stage2_pass.columns]])

display(novel_stage2_rescue[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_stage2_rescue.columns]])

=== Novel / ADC Stage 2 (P15) ===
Pass: 25
Rescue: 5


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
53,CLONE_0048,4.788019e-06,0.453842,10.695098,0.220395,1.751650e-05,0.523130,11.123578,0,0.698210,0.019093
54,CLONE_4878,3.345378e-06,0.497117,10.187778,0.176392,4.368307e-06,0.581270,9.401341,0,0.615499,0.016247
0,CLONE_1619,2.196463e-06,0.243030,3.403038,0.740186,2.452274e-06,0.153735,4.494834,1,0.922108,0.002972
55,CLONE_4665,1.320135e-06,0.704642,11.574957,0.061885,9.387567e-07,0.739219,9.048022,0,0.170636,0.097010
1,CLONE_3393,7.794316e-07,0.308014,8.439185,0.558652,7.938732e-07,0.365885,5.284487,0,0.566006,0.000000
56,CLONE_1505,5.616570e-07,0.519735,10.217237,0.010778,6.091065e-07,0.449628,11.902859,0,0.029747,0.004322
2,CLONE_0355,5.067278e-07,0.310332,7.402139,0.620693,4.564358e-07,0.421933,7.810760,0,0.215980,0.000000
57,CLONE_1035,4.939141e-07,0.335483,8.600502,0.540727,4.534849e-07,0.434488,8.809331,0,0.291253,0.000000
58,CLONE_4119,4.807651e-07,0.332187,3.761302,0.525444,4.881243e-07,0.299740,2.908559,1,0.240372,0.000000
3,CLONE_2358,4.670190e-07,0.294368,5.048017,0.666567,5.567181e-07,0.202139,4.869253,1,0.174645,0.000000


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
67,CLONE_4735,3.231743e-07,0.340824,2.658739,0.538190,3.113780e-07,0.389379,3.864801,0,0.086708,0.005631
11,CLONE_1150,3.174854e-07,0.307178,2.630148,0.681937,2.797309e-07,0.418127,2.875040,0,0.262892,0.000000
68,CLONE_4850,3.143818e-07,0.418493,11.845263,0.099822,2.600363e-07,0.520207,11.694074,0,0.017533,0.000000
12,CLONE_1834,3.138396e-07,0.285572,7.176047,0.659937,3.668220e-07,0.175000,6.882239,1,0.197058,0.002666
13,CLONE_1000,3.125661e-07,0.324422,6.084721,0.720187,3.653375e-07,0.242779,7.224040,1,0.318291,0.000000


In [12]:
# --------------------------------------------------
# Novel / ADC final selection
# --------------------------------------------------

novel_final_pool = pd.concat([novel_stage2_pass, novel_stage2_rescue], ignore_index=True)
novel_final_pool = novel_final_pool.sort_values(
    ["pred_late_qp", "pred_stable_prob"],
    ascending=[False, False]
).copy()

novel_top10 = novel_final_pool.head(TOP_N).copy()
novel_top3 = novel_final_pool.head(TOP_STAGE2).copy()

print("=== Novel / ADC Final Selection ===")
print("Final top10:", len(novel_top10))
print("Final top3:", len(novel_top3))

display(novel_top10[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_top10.columns]])

display(novel_top3[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in novel_top3.columns]])

=== Novel / ADC Final Selection ===
Final top10: 10
Final top3: 3


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_0048,4.788019e-06,0.453842,10.695098,0.220395,1.751650e-05,0.523130,11.123578,0,0.698210,0.019093
1,CLONE_4878,3.345378e-06,0.497117,10.187778,0.176392,4.368307e-06,0.581270,9.401341,0,0.615499,0.016247
2,CLONE_1619,2.196463e-06,0.243030,3.403038,0.740186,2.452274e-06,0.153735,4.494834,1,0.922108,0.002972
3,CLONE_4665,1.320135e-06,0.704642,11.574957,0.061885,9.387567e-07,0.739219,9.048022,0,0.170636,0.097010
4,CLONE_3393,7.794316e-07,0.308014,8.439185,0.558652,7.938732e-07,0.365885,5.284487,0,0.566006,0.000000
5,CLONE_1505,5.616570e-07,0.519735,10.217237,0.010778,6.091065e-07,0.449628,11.902859,0,0.029747,0.004322
6,CLONE_0355,5.067278e-07,0.310332,7.402139,0.620693,4.564358e-07,0.421933,7.810760,0,0.215980,0.000000
7,CLONE_1035,4.939141e-07,0.335483,8.600502,0.540727,4.534849e-07,0.434488,8.809331,0,0.291253,0.000000
8,CLONE_4119,4.807651e-07,0.332187,3.761302,0.525444,4.881243e-07,0.299740,2.908559,1,0.240372,0.000000
9,CLONE_2358,4.670190e-07,0.294368,5.048017,0.666567,5.567181e-07,0.202139,4.869253,1,0.174645,0.000000


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
0,CLONE_0048,0.000005,0.453842,10.695098,0.220395,0.000018,0.523130,11.123578,0,0.698210,0.019093
1,CLONE_4878,0.000003,0.497117,10.187778,0.176392,0.000004,0.581270,9.401341,0,0.615499,0.016247
2,CLONE_1619,0.000002,0.243030,3.403038,0.740186,0.000002,0.153735,4.494834,1,0.922108,0.002972


In [13]:
novel_eval_table = pd.DataFrame([
    eval_selection(novel_stage1_pass, "novel_P10_pass"),
    eval_selection(novel_stage1_rescue, "novel_P10_rescue"),
    eval_selection(novel_stage2_pass, "novel_P15_pass"),
    eval_selection(novel_stage2_rescue, "novel_P15_rescue"),
    eval_selection(novel_top10, "novel_final_top10"),
    eval_selection(novel_top3, "novel_final_top3"),
])

display(novel_eval_table)

novel_eval = {
    "selected_n": len(novel_top10),
    "mean_true_late_qp": float(novel_top10["true_late_qp"].mean()),
    "mean_true_qp_drop": float(novel_top10["true_qp_drop"].mean()),
    "mean_true_late_agg": float(novel_top10["true_late_agg"].mean()),
    "stable_rate": float(novel_top10["true_stable_label"].mean()),
    "top10_overlap_true_util_novel": topk_overlap(df["true_util_novel"], df["pred_util_novel"], TOP_N),
}
pd.DataFrame([novel_eval])

,label,n,mean_true_late_qp,mean_true_qp_drop,mean_true_late_agg,stable_rate
0,novel_P10_pass,53,2.809923e-07,0.282111,6.136780,0.566038
1,novel_P10_rescue,20,1.456370e-06,0.474628,9.827119,0.100000
2,novel_P15_pass,25,1.375791e-06,0.380228,7.920505,0.240000
3,novel_P15_rescue,5,3.166609e-07,0.349098,6.508039,0.400000
4,novel_final_top10,10,2.863358e-06,0.417117,7.565303,0.300000
5,novel_final_top3,3,8.112361e-06,0.419378,8.339918,0.333333


,selected_n,mean_true_late_qp,mean_true_qp_drop,mean_true_late_agg,stable_rate,top10_overlap_true_util_novel
0,10,0.000003,0.417117,7.565303,0.3,0.2


## Step 7 — Compare decision modes

This step makes it easier to see how the selected clone sets differ
between biosimilar mode and novel / ADC mode.

In [14]:
mode_compare = pd.DataFrame([
    {"mode": "biosimilar", **bio_eval},
    {"mode": "novel/ADC", **novel_eval},
])

display(mode_compare)

bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

print("Overlap between selected clone sets:", len(bio_ids & novel_ids))
print("Biosimilar-only:", sorted(list(bio_ids - novel_ids))[:10])
print("Novel-only:", sorted(list(novel_ids - bio_ids))[:10])

,mode,selected_n,mean_true_late_qp,mean_true_qp_drop,mean_true_late_agg,stable_rate,top10_overlap_true_util_bio,top10_overlap_true_util_novel
0,biosimilar,10,7.632078e-07,0.308293,7.225318,0.4,0.4,NaN
1,novel/ADC,10,2.863358e-06,0.417117,7.565303,0.3,NaN,0.2


Overlap between selected clone sets: 6
Biosimilar-only: ['CLONE_0151', 'CLONE_2759', 'CLONE_3602', 'CLONE_4484']
Novel-only: ['CLONE_0048', 'CLONE_1505', 'CLONE_4665', 'CLONE_4878']


## Step 8 — Decision validation block

This section validates whether the decision engine behaves reasonably before we move on to generator realism updates.

We check five things:

1. **Mode separation**
   - Are biosimilar and novel/ADC selections meaningfully different?

2. **Rescue volume**
   - Are rescue clone counts reasonable, rather than excessively large?

3. **Final selection quality**
   - Are the final Top-10 / Top-3 sets acceptably stable?

4. **Utility alignment**
   - Does predicted selection still overlap with true-utility winners?

5. **Pre-generator baseline**
   - These values serve as the baseline before applying late-aggregation realism patches.

This is the main validation step for the decision engine itself.

In [15]:
# --------------------------------------------------
# Step 8 — Decision validation block
# --------------------------------------------------

validation_rows = []

# ---------------------------
# Selection overlap / mode separation
# ---------------------------
bio_ids = set(bio_top10["clone_id"])
novel_ids = set(novel_top10["clone_id"])

top10_overlap_count = len(bio_ids & novel_ids)
top10_overlap_frac = top10_overlap_count / max(1, TOP_N)

# ---------------------------
# Rescue counts
# ---------------------------
bio_rescue_n_stage1 = len(bio_stage1_rescue) if "bio_stage1_rescue" in globals() else np.nan
bio_rescue_n_stage2 = len(bio_stage2_rescue) if "bio_stage2_rescue" in globals() else np.nan

novel_rescue_n_stage1 = len(novel_stage1_rescue) if "novel_stage1_rescue" in globals() else np.nan
novel_rescue_n_stage2 = len(novel_stage2_rescue) if "novel_stage2_rescue" in globals() else np.nan

# ---------------------------
# Final selection quality
# ---------------------------
bio_top10_stable_rate = float(bio_top10["true_stable_label"].mean())
bio_top3_stable_rate = float(bio_top3["true_stable_label"].mean()) if "bio_top3" in globals() else np.nan

novel_top10_stable_rate = float(novel_top10["true_stable_label"].mean())
novel_top3_stable_rate = float(novel_top3["true_stable_label"].mean()) if "novel_top3" in globals() else np.nan

# ---------------------------
# Utility overlap
# ---------------------------
bio_util_overlap = topk_overlap(df["true_util_bio"], df["pred_util_bio"], TOP_N)
novel_util_overlap = topk_overlap(df["true_util_novel"], df["pred_util_novel"], TOP_N)

# ---------------------------
# Validation table
# ---------------------------
validation_rows.append({
    "check": "mode_overlap_top10",
    "biosimilar": np.nan,
    "novel_adc": np.nan,
    "combined": top10_overlap_count,
    "notes": f"{top10_overlap_frac:.2f} of final top10 overlaps between modes",
})

validation_rows.append({
    "check": "stage1_rescue_n",
    "biosimilar": bio_rescue_n_stage1,
    "novel_adc": novel_rescue_n_stage1,
    "combined": np.nan,
    "notes": "Too high may indicate thresholds are too loose",
})

validation_rows.append({
    "check": "stage2_rescue_n",
    "biosimilar": bio_rescue_n_stage2,
    "novel_adc": novel_rescue_n_stage2,
    "combined": np.nan,
    "notes": "Final rescue count should remain limited",
})

validation_rows.append({
    "check": "final_top10_stable_rate",
    "biosimilar": bio_top10_stable_rate,
    "novel_adc": novel_top10_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better",
})

validation_rows.append({
    "check": "final_top3_stable_rate",
    "biosimilar": bio_top3_stable_rate,
    "novel_adc": novel_top3_stable_rate,
    "combined": np.nan,
    "notes": "Higher is better; should usually be >= top10 stable rate",
})

validation_rows.append({
    "check": "top10_true_utility_overlap",
    "biosimilar": bio_util_overlap,
    "novel_adc": novel_util_overlap,
    "combined": np.nan,
    "notes": "Ranking alignment with true utility",
})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,check,biosimilar,novel_adc,combined,notes
0,mode_overlap_top10,NaN,NaN,6.0,0.60 of final top10 overlaps between modes
1,stage1_rescue_n,9.000000,20.000000,NaN,Too high may indicate thresholds are too loose
2,stage2_rescue_n,NaN,5.000000,NaN,Final rescue count should remain limited
3,final_top10_stable_rate,0.400000,0.300000,NaN,Higher is better
4,final_top3_stable_rate,0.666667,0.333333,NaN,Higher is better; should usually be >= top10 s...
5,top10_true_utility_overlap,0.400000,0.200000,NaN,Ranking alignment with true utility


In [16]:
print("=== Decision validation summary ===")
print(f"Final top10 overlap between biosimilar and novel/ADC: {top10_overlap_count}/{TOP_N} ({top10_overlap_frac:.2%})")

print("\nRescue counts")
print(f"- Biosimilar: Stage1={bio_rescue_n_stage1}, Stage2={bio_rescue_n_stage2}")
print(f"- Novel/ADC: Stage1={novel_rescue_n_stage1}, Stage2={novel_rescue_n_stage2}")

print("\nFinal stable rates")
print(f"- Biosimilar: top10={bio_top10_stable_rate:.3f}, top3={bio_top3_stable_rate:.3f}")
print(f"- Novel/ADC: top10={novel_top10_stable_rate:.3f}, top3={novel_top3_stable_rate:.3f}")

print("\nUtility overlap")
print(f"- Biosimilar top10 utility overlap: {bio_util_overlap:.3f}")
print(f"- Novel/ADC top10 utility overlap: {novel_util_overlap:.3f}")

=== Decision validation summary ===
Final top10 overlap between biosimilar and novel/ADC: 6/10 (60.00%)

Rescue counts
- Biosimilar: Stage1=9, Stage2=nan
- Novel/ADC: Stage1=20, Stage2=5

Final stable rates
- Biosimilar: top10=0.400, top3=0.667
- Novel/ADC: top10=0.300, top3=0.333

Utility overlap
- Biosimilar top10 utility overlap: 0.400
- Novel/ADC top10 utility overlap: 0.200


## Step 9 — Utility-based weight sweep (optional analysis)

This section is not the main deployment rule.

Instead, it helps us explore whether different utility weight combinations
could improve top-k clone selection.

This is useful for:
- comparing biosimilar vs novel priorities
- tuning future SDL decision rules
- stress-testing the decision engine

In [17]:
grid_bio = {
    "w_qp": [0.5, 1.0, 1.5, 2.0],
    "w_drop": [0.5, 1.0, 1.5, 2.0],
    "w_agg": [0.0, 0.1, 0.2, 0.3],
}

grid_novel = {
    "w_qp": [0.5, 1.0, 1.5],
    "w_drop": [0.5, 1.0, 1.5],
    "w_agg": [0.5, 1.0, 1.5, 2.0],
}

ELITE_FRACS = [0.10]
K_LIST = [5, 10, 20]

def mark_elite(df, util_col, frac):
    thr = df[util_col].quantile(1 - frac)
    return (df[util_col] >= thr).astype(int)

def run_sweep(base_df, mode_name, grid, true_util_col):
    rows = []
    work = base_df.copy()

    for w_qp in grid["w_qp"]:
        for w_drop in grid["w_drop"]:
            for w_agg in grid["w_agg"]:
                work["pred_util_tmp"] = (
                    w_qp * z(work["pred_late_qp"])
                    - w_drop * z(work["pred_qp_drop"])
                    - w_agg * z(work["pred_late_agg"])
                )

                for frac in ELITE_FRACS:
                    elite_col = f"elite_{int(frac*100)}"
                    work[elite_col] = mark_elite(work, true_util_col, frac)

                    for k in K_LIST:
                        rows.append({
                            "mode": mode_name,
                            "w_qp": w_qp,
                            "w_drop": w_drop,
                            "w_agg": w_agg,
                            "elite_frac": frac,
                            "k": k,
                            "precision@k": precision_at_k_df(work, "pred_util_tmp", elite_col, k),
                            "ndcg@k": ndcg_at_k_df(work, "pred_util_tmp", elite_col, k),
                        })
    return pd.DataFrame(rows)

bio_sweep = run_sweep(df, "biosimilar", grid_bio, "true_util_bio")
novel_sweep = run_sweep(df, "novel/ADC", grid_novel, "true_util_novel")

display(bio_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))
display(novel_sweep.sort_values(["precision@k", "ndcg@k"], ascending=False).head(10))

,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
0,biosimilar,0.5,0.5,0.0,0.1,5,1.0,1.0
3,biosimilar,0.5,0.5,0.1,0.1,5,1.0,1.0
6,biosimilar,0.5,0.5,0.2,0.1,5,1.0,1.0
9,biosimilar,0.5,0.5,0.3,0.1,5,1.0,1.0
12,biosimilar,0.5,1.0,0.0,0.1,5,1.0,1.0
15,biosimilar,0.5,1.0,0.1,0.1,5,1.0,1.0
18,biosimilar,0.5,1.0,0.2,0.1,5,1.0,1.0
21,biosimilar,0.5,1.0,0.3,0.1,5,1.0,1.0
33,biosimilar,0.5,1.5,0.3,0.1,5,1.0,1.0
36,biosimilar,0.5,2.0,0.0,0.1,5,1.0,1.0


,mode,w_qp,w_drop,w_agg,elite_frac,k,precision@k,ndcg@k
9,novel/ADC,0.5,0.5,2.0,0.1,5,0.8,0.955830
18,novel/ADC,0.5,1.0,1.5,0.1,5,0.8,0.904717
21,novel/ADC,0.5,1.0,2.0,0.1,5,0.8,0.904717
30,novel/ADC,0.5,1.5,1.5,0.1,5,0.8,0.904717
34,novel/ADC,0.5,1.5,2.0,0.1,10,0.7,0.941451
31,novel/ADC,0.5,1.5,1.5,0.1,10,0.7,0.898119
19,novel/ADC,0.5,1.0,1.5,0.1,10,0.7,0.897150
29,novel/ADC,0.5,1.5,1.0,0.1,20,0.7,0.875302
28,novel/ADC,0.5,1.5,1.0,0.1,10,0.7,0.874360
25,novel/ADC,0.5,1.5,0.5,0.1,10,0.7,0.776201


## Step 10 — Identify borderline but potentially rescuable clones

These are clones that:

- have strong predicted productivity
- are close to passing the quality or stability threshold
- may be interesting for future process-rescue strategies

This section is exploratory, but it is important for long-term SDL design.

In [18]:
# Example: near-pass clones
MARGIN_AGG = 2.0
MARGIN_STABLE = 0.05

borderline = df.copy()

borderline["high_prod"] = borderline["pred_late_qp"] >= borderline["pred_late_qp"].quantile(0.90)
borderline["near_quality"] = borderline["pred_late_agg"].between(BIO_AGG_MAX, BIO_AGG_MAX + MARGIN_AGG)
borderline["near_stability"] = borderline["pred_stable_prob"].between(BIO_STABLE_PROB_MIN - MARGIN_STABLE, BIO_STABLE_PROB_MIN)

resc_candidates = borderline[
    borderline["high_prod"] & (borderline["near_quality"] | borderline["near_stability"])
].copy()

resc_candidates = resc_candidates.sort_values("pred_late_qp", ascending=False)

print("Potentially rescuable candidates:", len(resc_candidates))
display(resc_candidates[[
    "clone_id",
    "pred_late_qp", "pred_qp_drop", "pred_late_agg", "pred_stable_prob",
    "true_late_qp", "true_qp_drop", "true_late_agg", "true_stable_label"
] + [c for c in ["pred_super_prob", "pred_aggr_prob"] if c in resc_candidates.columns]].head(20))

Potentially rescuable candidates: 1


,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_stable_prob,true_late_qp,true_qp_drop,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob
966,CLONE_0151,3.820308e-07,0.34452,5.656948,0.472694,3.087592e-07,0.450622,6.537014,0,0.055487,0.0


## Output of Notebook 04b

This notebook converts predicted late-stage outcomes into actual clone selection decisions.

### Main outputs
- biosimilar-mode selected clones
- novel / ADC-mode selected clones
- offline evaluation against synthetic truth
- utility-based comparison of scoring rules
- exploratory list of borderline / rescuable candidates
- decision validation summary
  - mode separation
  - rescue volume
  - final stable rate
  - utility overlap

### Pipeline meaning
- **Notebook 03b = prediction engine**
- **Notebook 04b = decision engine**

This separation makes the workflow more extensible for future SDL systems, where:
- thresholds may change
- product profiles may differ
- rescue strategies may be added without retraining the prediction model